## Using headers to bypass bot detection
This mimics a human


In [ ]:
import requests

# response = requests.get("https://yamahamusicstore.in/")

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
}
# Use headers to bypass bot detection

response = requests.get("https://yamahamusicstore.in/", headers=headers, timeout=10)

print(response.status_code)

In [ ]:
html_doc = response.text
html_doc

## Converting the HTML String to Beautiful Soup Object

In [ ]:
from bs4 import BeautifulSoup
soup = BeautifulSoup(html_doc, 'html.parser')

print(soup.prettify())

In [ ]:
categories_section = soup.find("div",class_="flat-animate-tab")
categories_section

In [ ]:
all_products = categories_section.find_all("div", class_="catgcvr")
all_products

In [ ]:
all_products[0].find("a").get("href")


In [ ]:
all_products[0].get_text(strip=True)

In [ ]:
categories = {}

for product in all_products:
    print(product, "\n")
    link = product.find("a").get("href")
    category = product.get_text(strip=True)
    categories[category]=link
categories


## Getting items of each category

In [ ]:
category_response = requests.get(categories['Keyboard Instruments'], headers=headers, timeout=10)
category_response.status_code

In [ ]:
category_html = category_response.text
category_soup = BeautifulSoup(category_html, 'html.parser')
print(soup.prettify)

In [ ]:
products_container = category_soup.find("div", class_="tf-list-layout wrapper-shop prodLising")
products_container

In [ ]:
products_cards = products_container.find_all("div", class_="card-product-info")
products_cards

## Cleaning for one product card

In [ ]:
card_link = products_cards[0].find("a").get("href")

product_name = products_cards[0].find("a").get_text(strip=True)

card_link = products_cards[0].find("div", class_="list-star")
if not card_link:
    print(f"No rating found for product {product_name} on page {page} for category {category}")
    continue

rating = card_link.get_text(strip=True)

price_section = products_cards[0].find("div", class_="price")
price = price_section.get_text(strip=True)

try:
    review_tag =products_cards[0].find("div", class_="text text-caption-1")
    review_count = review_tag.get_text(strip=True)
except Exception as e:
    review_count = "(0 reviews)"

all_page_product_card_info["keyboard_instruments"].append({
    "name":product_name,
    "rating": rating,
    "price": price,
    "review_count": review_count
})


In [ ]:
product_name =products_cards[0].find("a").get_text(strip=True)
product_name

In [ ]:
card_link = products_cards[2].find("div", class_="list-star")
rating= card_link.get_text(strip=True)
rating

In [ ]:
price_section=products_cards[0].find("div", class_="price")
price = price_section.get_text(strip=True)
price

In [ ]:
products_cards[-2]

In [ ]:
try:
    review_tag =products_cards[0].find("div", class_="text text-caption-1")
    review_count = review_tag.get_text(strip=True)
except Exception as e:
    review_count = "(0 reviews)"
    
    
review_count

In [ ]:
product_card_info = {}

for card in products_cards:

    card_link=card.find("a").get("href")
    product_name =card.find("a").get_text(strip=True)
    
    card_link = card.find("div", class_="list-star")
    rating= card_link.get_text(strip=True)
    
    price_section=card.find("div", class_="price")
    price = price_section.get_text(strip=True)
    
    try:
        review_tag =card.find("div", class_="text text-caption-1")
        review_count = review_tag.get_text(strip=True)
    except Exception as e:
        review_count = "(0 reviews)"
        
    product_card_info[product_name] = {
         "name":product_name,
         "rating": rating,
        "price": price,
        "review_count": review_count
    }
    


In [ ]:
product_card_info

## For Image Scraping

In [ ]:
img_url = products_container.find("img").get("src")
img_url

In [ ]:
from urllib.parse import urlparse

parsed_url = urlparse(img_url)
 
parsed_url

In [ ]:
import os
ext = os.path.splitext(parsed_url.path)[1]
ext

In [ ]:
import os
from urllib.parse import urlparse

try:
    img_data = requests.get(img_url, headers=headers, stream=True)
    img_data.raise_for_status()

    # 1. Get the correct file extension from the image URL (.jpg, .png, etc.)
    parsed_url = urlparse(img_url)
    ext = os.path.splitext(parsed_url.path)[1]

    # Fallback to .jpg if the URL doesn't contain a clear extension
    if not ext:
        ext = ".jpg"

    # 2. Set your directory path and choose a filename (e.g., image_1.jpg)
    folder_path = r"/home/chris/Documents/Broadway Python/Python Project I"
    filename = f"downloaded_image{ext}"  # Or use a dynamic variable like f"img_{i}{ext}"
    full_file_path = os.path.join(folder_path, filename)

    # 3. Save the image safely
    with open(full_file_path, "wb") as file:
        for chunk in img_data.iter_content(chunk_size=8192):
            file.write(chunk)

    print(f"Successfully downloaded to {full_file_path}")

except Exception as e:
    print(f"Skipping {img_url} due to error: {e}")


## For other pages

In [ ]:
target_pages = 5

all_page_product_card_info = {
    'keyboard_instruments': []
}

for category, link in categories.items():
    for page in range(1, target_pages+1):
        dynamic_url = f"{link}?page={page}"
        
        html_doc = requests.get(dynamic_url, headers=headers, timeout=10)
        if html_doc.status_code !=200:
            break


        category_soup = BeautifulSoup(html_doc.text, 'html.parser')

        products_container = category_soup.find("div", class_="tf-list-layout wrapper-shop prodLising")
        print(f"Scraping page {page} for category {category}")
        if not products_container:
            print(f"No products container found on page {page} for category {category}")
            continue

        products_cards = products_container.find_all("div", class_="card-product-info")
        if not products_cards:
            print(f"No products found in products_cards on page {page} for category {category}")
            continue
        card_link = products_cards[0].find("a").get("href")
        
        product_name = products_cards[0].find("a").get_text(strip=True)

        card_link = products_cards[0].find("div", class_="list-star")
        if not card_link:
            print(f"No rating found for product {product_name} on page {page} for category {category}")
            continue
        
        rating = card_link.get_text(strip=True)

        price_section = products_cards[0].find("div", class_="price")
        price = price_section.get_text(strip=True)

        try:
            review_tag =products_cards[0].find("div", class_="text text-caption-1")
            review_count = review_tag.get_text(strip=True)
        except Exception as e:
            review_count = "(0 reviews)"

        all_page_product_card_info["keyboard_instruments"].append({
            "name":product_name,
            "rating": rating,
            "price": price,
            "review_count": review_count
        })



In [ ]:
all_page_product_card_info